# Лекција 05 - Агентни RAG


## Подешавање

Овај бележник демонстрира Agentic RAG (Retrieval-Augmented Generation) образац користећи Microsoft Agent Framework.

**Пре захтева:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ваш крајњи узлазак Azure AI Search сервиса
- `AZURE_SEARCH_API_KEY` — ваш Azure AI Search API кључ
- Azure OpenAI имплементација конфигурисана преко променљивих окружења
- Azure CLI аутентификован (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Шта је Agentic RAG?

Традиционални RAG следи фиксни ток: прво преузима документе, а затим генерише одговор. **Agentic RAG** иде даље тако што агенту даје аутономију да одлучи **када** и **како** да преузме информације.

Са Agentic RAG, агент може:
- **Одлучити** да ли је преузимање података потребно пре одговора на питање
- **Изабрати** који извор података или алат да упита
- **Проценити** резултате преузимања и извршити додатна преузимања ако је први покушај недовољан
- **Комбиновати** информације из више корака преузимања у кохерентан одговор

Ово чини агента флексибилнијим и прецизнијим у поређењу са статичним процесом преузми-затим-генериши.


## Креирање алата за претрагу

У Agentic RAG, спољашњи извори података су омотани као **алати** које агент може позвати по потреби. Ово омогућава агенту да третира претрагу као још једну акцију коју може извршити, уместо као обавезан корак.

Испод дефинишемо базу знања о путовањима и излажемо је као алат који агент може користити за проналажење информација о дестинацијама.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Креирање RAG агента

Сада правимо агента који је упућен да **увек прво преузме информације пре него што одговори**. Агент користи алатку `search_travel_knowledge` да би темељио своје одговоре на бази знања, уместо да се ослања на своје обучавајуће податке.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Итертивно Претраживање — Образац Прављач-Проверавач

Кључна предност Agentic RAG је **итертивно претраживање**. агент може извршити више рунди претраге да би верификовао, уредио или проширио своје почетне налазе — слично као радни ток „прављач-проверавач“:

1. **Прављачки корак**: агент преузима почетне информације и саставља одговор.
2. **Проверни корак**: агент извршава додатне претраге да би проверио детаље или попунио празнине.

Испод, агенту је постављено питање које захтева упоређивање више дестинација, што га подстиче да више пута претражује.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Сажетак

У овој лекцији сте научили како да направите систем **Agentic RAG** користећи Microsoft Agent Framework:

- **Agentic RAG** омогућава агентима да аутономно одлучују када да преузму информације, чинећи преузимање динамичним уместо фиксним.
- **Алатке као извори података**: Спољне базе знања (као што је Azure AI Search) су упаковане као алатке које агент може да позове.
- **Итеративно преузимање**: Шема maker-checker омогућава агенту да изврши више рунди преузимања — претражујући, проверавајући и усавршавајући — пре него што произведе коначни одговор.

У продукцији бисте заменили ин-мемори `TRAVEL_KNOWLEDGE_BASE` са стварним Azure AI Search индексом да бисте руковали преузимањем великих количина докумената о путовањима.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
